In [11]:
# ============================================================
# DRONECROWD V9 — CrowdNet (edge-friendly, grayscale)
# ============================================================
# Corrected pipeline addressing V7/V8 failures.
#
# WHAT CHANGED vs V8 (and WHY):
#   1. NATIVE-RES CROPS, not whole-frame resize.
#      V8 resized 1920x1080 -> 448 (still a 4.3x downscale), so a
#      ~10px head became ~2px. We keep native pixels and train on
#      512x512 crops; test full-frame (model is fully-convolutional).
#   2. MobileNetV3-Small converted to OUTPUT-STRIDE-8 via DILATION.
#      Gives 576-ch semantic features AT 1/8 resolution, single
#      stream, no FPN, no ASPP pyramid (those added random params
#      and noise). This is the CSRNet philosophy on a mobile backbone.
#   3. BatchNorm RESTORED in the density head (V8 removed it -> the
#      random-init dilated head was badly conditioned -> worse MAE).
#   4. Small fixed-sigma GT built at NATIVE res, downsampled to 1/8
#      by a count-preserving reshape-sum. MSE (Euclidean) loss.
#   5. Grayscale, [0,1] input (matches the synthetic weights). NO
#      ImageNet mean/std normalization.
#   6. Temporal head kept from synthetic init, UNFROZEN (Stage 2),
#      with a frozen synthetic snapshot used for Task B.
#
# RUN ORDER (bottom driver cell):
#   preprocess_all() -> sanity -> train Stage 1 -> Task A/B/C
#   Optionally flip USE_TEMPORAL_COUNT=True and run Stage 2.
# ============================================================

import os, re, glob, math, random, json, copy
from collections import defaultdict

import cv2
cv2.setNumThreads(0)
import numpy as np
import scipy.io
import scipy.ndimage
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
import torchvision.models as models
from torch.utils.data import Dataset, DataLoader
from tqdm.auto import tqdm

torch.manual_seed(42); np.random.seed(42); random.seed(42)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("=" * 60); print(f"DEVICE : {device}"); print("=" * 60)

# ---------------- PATHS ----------------
DRONECROWD_ROOT = "dronecrowd"
PROCESSED_DIR   = "dronecrowd_processed_v9"
SYNTH_CKPT      = "best_crowd_model.pth"                 # synthetic grayscale weights
MOBILENET_LOCAL = "mobilenet_v3_small-047dcff4.pth"     # optional local backbone
BEST_MODEL      = "best_dronecrowd_v9.pth"

# ---------------- GEOMETRY ----------------
IMG_W, IMG_H  = 1920, 1080
OUT_STRIDE    = 8
DEN_W, DEN_H  = IMG_W // OUT_STRIDE, IMG_H // OUT_STRIDE   # 240 x 135
CROP          = 512                                        # native crop (multiple of 8)
CROP_D        = CROP // OUT_STRIDE                         # 64  (density crop size)
TEMP_SIZE     = 224                                        # temporal branch input (matches synthetic)
DENSITY_SIGMA_NATIVE = 4.0                                 # small fixed sigma at native res

# ---------------- WINDOWS (temporal only) ----------------
MIN_FRAMES      = 20
SEQ_LENGTH      = 10
STRIDE_W        = 5
WARMUP_FRAMES   = 3
VAL_DATA_FRAMES = 12
VAL_SEQ_LENGTH  = 8

# ---------------- TRAINING ----------------
EPOCHS             = 60
BATCH_SIZE         = 8          # 512-crop density batch; lower if OOM
NUM_WORKERS        = 0
TRAIN_FRAME_STRIDE = 5          # use every Nth frame per seq (300fps-redundant video)
LR                 = 1e-4
WEIGHT_DECAY       = 5e-4
PATIENCE           = 12
COUNT_LOSS_WEIGHT  = 0.02

# ---------------- TEMPORAL (Stage 2) ----------------
USE_TEMPORAL_COUNT = False      # STAGE 1: False (spatial only). STAGE 2: True.
TEMP_SMOOTH_WEIGHT = 0.10

# ---------------- TASK B/C ----------------
CROWDED_THRESH  = 150
CLASS_NAMES     = {0: "Normal", 1: "Bottleneck", 2: "Panic"}
TASK_C_MAX_SEQS = None
TASK_C_MAX_FRAMES = None

# ---------------- RUN MODE ----------------
MAX_SEQS_SMOKE = None           # None = full run; e.g. 3 = quick smoke test

DEVICE : cuda


In [2]:
# ============================================================
# STEP 1 : FILENAME / ANNOTATION UTILITIES
# ============================================================

def parse_image_name(fname):
    m = re.match(r"(?:GT_)?img(\d{3})(\d{3})(?:\.jpg|\.mat)?$", os.path.basename(fname))
    return (int(m.group(1)), int(m.group(2))) if m else (None, None)

def image_name(seq_id, frame_id): return f"img{seq_id:03d}{frame_id:03d}.jpg"
def gt_name(seq_id, frame_id):    return f"GT_img{seq_id:03d}{frame_id:03d}.mat"

def read_mat(mat_path):
    mat  = scipy.io.loadmat(mat_path)
    cell = mat["image_info"][0, 0]
    loc  = cell["location"][0, 0].astype(np.float32)     # (N,>=2): x, y, ...
    return loc[:, :2], int(cell["number"][0, 0].item())

def jpg_path(split_disk, seq_id, fid):
    return os.path.join(DRONECROWD_ROOT, split_disk, "images", image_name(seq_id, fid))


# ============================================================
# STEP 2 : DENSITY MAP  (native sigma -> 1/8 by count-preserving sum)
# ============================================================

def build_density_native(points_xy):
    # points_xy: (N,2) head centres in NATIVE 1920x1080 coordinates.
    # Returns (DEN_H, DEN_W) float16 map whose sum == N (exactly).
    d = np.zeros((IMG_H, IMG_W), np.float32)
    for x, y in points_xy:
        d[int(np.clip(y, 0, IMG_H - 1)), int(np.clip(x, 0, IMG_W - 1))] += 1.0
    d = scipy.ndimage.gaussian_filter(d, sigma=DENSITY_SIGMA_NATIVE)
    d = d.reshape(DEN_H, OUT_STRIDE, DEN_W, OUT_STRIDE).sum(axis=(1, 3))
    return d.astype(np.float16)


# ============================================================
# STEP 3 : DISCOVER SEQUENCES
# ============================================================

def discover_sequences(split_disk, min_frames):
    img_dir = os.path.join(DRONECROWD_ROOT, split_disk, "images")
    if not os.path.exists(img_dir):
        print(f"  [{split_disk}] images/ not found"); return {}
    seq_frames = defaultdict(list)
    for fname in os.listdir(img_dir):
        sid, fid = parse_image_name(fname)
        if sid is not None: seq_frames[sid].append(fid)
    seq_dict = {sid: sorted(f) for sid, f in seq_frames.items() if len(f) >= min_frames}
    print(f"  [{split_disk}] {len(seq_dict)} sequences "
          f"({sum(len(v) for v in seq_dict.values())} frames)  [min_frames={min_frames}]")
    return seq_dict


# ============================================================
# STEP 4 : PRE-PROCESS  (stores density_ / gt_counts / trajectories / frame_ids)
#          NOTE: frames are NOT copied — we read the original jpgs at load time
#          and crop at native resolution.  Only lightweight labels are stored.
# ============================================================

def preprocess_split(split_disk, min_frames):
    gt_dir   = os.path.join(DRONECROWD_ROOT, split_disk, "ground_truth")
    seq_dict = discover_sequences(split_disk, min_frames)
    if not seq_dict: return
    seq_ids = sorted(seq_dict.keys())
    if MAX_SEQS_SMOKE is not None:
        seq_ids = seq_ids[:MAX_SEQS_SMOKE]
        print(f"  SMOKE: {len(seq_ids)} seqs")

    for seq_id in tqdm(seq_ids, desc=f"  preprocess {split_disk}"):
        frame_ids = seq_dict[seq_id]
        out_seq   = os.path.join(PROCESSED_DIR, split_disk, f"seq{seq_id:03d}")
        done_flag = os.path.join(out_seq, "gt_counts.npy")
        if os.path.exists(done_flag) and \
           len(glob.glob(os.path.join(out_seq, "density_*.npy"))) == len(frame_ids):
            continue
        os.makedirs(out_seq, exist_ok=True)

        gt_counts, traj_rows = [], []
        for t, fid in enumerate(frame_ids):
            mp = os.path.join(gt_dir, gt_name(seq_id, fid))
            if os.path.exists(mp):
                pts, cnt = read_mat(mp)
            else:
                pts, cnt = np.zeros((0, 2), np.float32), 0
            gt_counts.append(float(cnt))
            np.save(os.path.join(out_seq, f"density_{t:03d}.npy"), build_density_native(pts))
            for (x, y) in pts:
                traj_rows.append([float(t), float(x), float(y)])

        np.save(os.path.join(out_seq, "gt_counts.npy"),  np.array(gt_counts, np.float32))
        np.save(os.path.join(out_seq, "frame_ids.npy"),  np.array(frame_ids, np.int32))
        if traj_rows:
            np.save(os.path.join(out_seq, "trajectories.npy"), np.array(traj_rows, np.float32))


def preprocess_all():
    os.makedirs(PROCESSED_DIR, exist_ok=True)
    for split_disk, mf in [("train_data", MIN_FRAMES), ("test_data", MIN_FRAMES),
                           ("val_data", VAL_DATA_FRAMES)]:
        if not os.path.exists(os.path.join(DRONECROWD_ROOT, split_disk)):
            print(f"WARNING: {split_disk} not found — skipping"); continue
        print(f"\n{'='*50}\nPre-processing: {split_disk}\n{'='*50}")
        preprocess_split(split_disk, mf)
    print("\nPre-processing complete.")


# ============================================================
# SANITY : density sum must equal GT count
# ============================================================

def sanity_check(n_frames=6):
    ok_all = True
    for split_disk in ["train_data", "val_data", "test_data"]:
        root = os.path.join(PROCESSED_DIR, split_disk)
        if not os.path.exists(root): continue
        seqs = sorted(os.listdir(root))
        if not seqs: continue
        sp = os.path.join(root, seqs[0])
        gt = np.load(os.path.join(sp, "gt_counts.npy"))
        print(f"\n{'='*54}\nSANITY [{split_disk}] {seqs[0]}")
        print(f"{'Frame':>6}{'GT':>10}{'MapSum':>10}{'Diff':>9}  ok")
        for t in range(min(n_frames, len(gt))):
            d = np.load(os.path.join(sp, f"density_{t:03d}.npy")).astype(np.float32)
            s = float(d.sum()); g = float(gt[t])
            good = abs(s - g) < max(2.0, g * 0.02); ok_all &= good
            print(f"{t:>6}{g:>10.1f}{s:>10.2f}{s-g:>+9.2f}  {'Y' if good else 'N'}")
    print(f"\nSanity: {'PASSED' if ok_all else 'FAILED'}")
    if not ok_all: raise RuntimeError("Density sanity failed.")

In [3]:
# ============================================================
# AUGMENTATION  (photometric only — keeps head positions, so the
# density target stays valid.  NO flips/geometry.)
# ============================================================

def photometric_aug(img, p=0.5):
    if np.random.rand() > p: return img
    mode = np.random.choice(["gauss", "blur", "bright", "sp", "fog", "none"],
                            p=[0.22, 0.16, 0.18, 0.16, 0.14, 0.14])
    f = img.astype(np.float32)
    if mode == "gauss":
        f = f + np.random.normal(0, np.random.uniform(5, 20), img.shape)
    elif mode == "blur":
        k = int(np.random.choice([3, 5, 7])); return cv2.GaussianBlur(img, (k, k), 0)
    elif mode == "bright":
        f = f * np.random.uniform(0.5, 1.6)
    elif mode == "sp":
        a = img.copy(); m = np.random.rand(*img.shape); pr = np.random.uniform(0.005, 0.04)
        a[m < pr/2] = 0; a[(m >= pr/2) & (m < pr)] = 255; return a
    elif mode == "fog":
        al = np.random.uniform(0.15, 0.45); f = (1-al)*f + al*128
    else:
        return img
    return np.clip(f, 0, 255).astype(np.uint8)


SPLIT_DISK = {"train": "train_data", "val": "val_data", "test": "test_data"}


# ============================================================
# DATASET A : PER-FRAME DENSITY  (Stage 1 + Task A)
#   training=True  -> random 512x512 native crop  (count = crop density sum)
#   training=False -> full 1920x1080 frame        (count = true GT count)
# ============================================================

class FrameDensityDataset(Dataset):
    def __init__(self, split, training=False, frame_stride=1, aug_prob=0.5):
        self.split_disk = SPLIT_DISK[split]
        self.training   = training
        self.aug_prob   = aug_prob
        self.items      = []                       # (seq_dir, seq_id, t, fid)
        root = os.path.join(PROCESSED_DIR, self.split_disk)
        if not os.path.exists(root):
            raise FileNotFoundError(f"{root} — run preprocess_all() first.")
        for seq_name in sorted(os.listdir(root)):
            seq_dir = os.path.join(root, seq_name)
            fids = np.load(os.path.join(seq_dir, "frame_ids.npy"))
            seq_id = int(seq_name.replace("seq", ""))
            for t in range(0, len(fids), frame_stride):
                self.items.append((seq_dir, seq_id, t, int(fids[t])))
        print(f"  FrameDensity[{split}] {len(self.items)} frames "
              f"(training={training}, stride={frame_stride})")

    def __len__(self): return len(self.items)

    def _read_gray(self, seq_id, fid):
        img = cv2.imread(jpg_path(self.split_disk, seq_id, fid), cv2.IMREAD_GRAYSCALE)
        if img is None:
            img = np.zeros((IMG_H, IMG_W), np.uint8)
        elif img.shape != (IMG_H, IMG_W):
            img = cv2.resize(img, (IMG_W, IMG_H))
        return img

    def __getitem__(self, idx):
        seq_dir, seq_id, t, fid = self.items[idx]
        img = self._read_gray(seq_id, fid)
        den_full = np.load(os.path.join(seq_dir, f"density_{t:03d}.npy")).astype(np.float32)

        if self.training:
            x0 = 8 * np.random.randint(0, (IMG_W - CROP) // 8 + 1)
            y0 = 8 * np.random.randint(0, (IMG_H - CROP) // 8 + 1)
            img = img[y0:y0+CROP, x0:x0+CROP]
            den = den_full[y0//8:y0//8+CROP_D, x0//8:x0//8+CROP_D]
            img = photometric_aug(img, self.aug_prob)
            count = float(den.sum())
        else:
            den   = den_full
            gt    = np.load(os.path.join(seq_dir, "gt_counts.npy"))
            count = float(gt[t])

        frame = torch.from_numpy(img.astype(np.float32) / 255.0).unsqueeze(0)   # (1,H,W)
        dmap  = torch.from_numpy(den.copy()).unsqueeze(0)                        # (1,h,w)
        return frame, dmap, torch.tensor(count, dtype=torch.float32)


# ============================================================
# DATASET B : TEMPORAL WINDOWS  (Task B, and Stage 2 training)
#   Yields TEMP_SIZE (224) full-frame views + on-the-fly flow.
#   For Stage-2 counting it ALSO yields the aligned native crop
#   window + density so the temporal state can refine per-frame counts.
# ============================================================

def farneback(prev, cur):
    fl = cv2.calcOpticalFlowFarneback(prev, cur, None, 0.5, 3, 15, 3, 5, 1.2, 0)
    return np.clip(fl / 8.0, -1.0, 1.0).astype(np.float32)     # (H,W,2) in [-1,1]


class WindowDataset(Dataset):
    # mode="classify"  -> (frames224, flows224)                      [Task B]
    # mode="count"     -> (crops, flows224, densities, counts)       [Stage 2]
    def __init__(self, split, mode="classify", seq_length=SEQ_LENGTH,
                 stride=STRIDE_W, training=False):
        self.split_disk = SPLIT_DISK[split]
        self.mode = mode; self.training = training
        self.seq_length = VAL_SEQ_LENGTH if split == "val" else seq_length
        self.windows = []
        root = os.path.join(PROCESSED_DIR, self.split_disk)
        for seq_name in sorted(os.listdir(root)):
            seq_dir = os.path.join(root, seq_name)
            fids = np.load(os.path.join(seq_dir, "frame_ids.npy"))
            seq_id = int(seq_name.replace("seq", ""))
            n = len(fids)
            if n < self.seq_length + 1: continue
            if split == "val":
                starts = [0]
            else:
                starts = list(range(WARMUP_FRAMES, n - self.seq_length, stride))
            for s in starts:
                self.windows.append((seq_dir, seq_id, s, fids))
        print(f"  Window[{split}/{mode}] {len(self.windows)} windows (T={self.seq_length})")

    def __len__(self): return len(self.windows)

    def _read_gray(self, seq_id, fid):
        img = cv2.imread(jpg_path(self.split_disk, seq_id, fid), cv2.IMREAD_GRAYSCALE)
        if img is None: img = np.zeros((IMG_H, IMG_W), np.uint8)
        elif img.shape != (IMG_H, IMG_W): img = cv2.resize(img, (IMG_W, IMG_H))
        return img

    def __getitem__(self, idx):
        seq_dir, seq_id, s, fids = self.windows[idx]
        T = self.seq_length

        # choose one native crop box, shared across the window (temporal coherence)
        if self.mode == "count" and self.training:
            x0 = 8 * np.random.randint(0, (IMG_W - CROP) // 8 + 1)
            y0 = 8 * np.random.randint(0, (IMG_H - CROP) // 8 + 1)

        small, crops, dens, counts = [], [], [], []
        for k in range(T + 1):
            g = self._read_gray(seq_id, int(fids[s + k]))
            small.append(cv2.resize(g, (TEMP_SIZE, TEMP_SIZE)))
            if self.mode == "count" and k < T:
                den_full = np.load(os.path.join(seq_dir, f"density_{s+k:03d}.npy")).astype(np.float32)
                if self.training:
                    c = g[y0:y0+CROP, x0:x0+CROP]
                    d = den_full[y0//8:y0//8+CROP_D, x0//8:x0//8+CROP_D]
                    counts.append(float(d.sum()))
                else:
                    c = g; d = den_full   # NATIVE full frame (FCN) — no resolution loss at eval
                    gt = np.load(os.path.join(seq_dir, "gt_counts.npy")); counts.append(float(gt[s+k]))
                crops.append(torch.from_numpy(c.astype(np.float32)/255.).unsqueeze(0))
                dens.append(torch.from_numpy(d.copy()).unsqueeze(0))

        flows = [torch.from_numpy(farneback(small[t], small[t+1])).permute(2, 0, 1)
                 for t in range(T)]
        frames224 = torch.stack([torch.from_numpy(small[t].astype(np.float32)/255.).unsqueeze(0)
                                 for t in range(T)])
        flows224  = torch.stack(flows)

        if self.mode == "classify":
            return frames224, flows224
        return (torch.stack(crops), frames224, flows224,
                torch.stack(dens), torch.tensor(counts, dtype=torch.float32))

In [4]:
# ============================================================
# MODEL BUILDING BLOCKS
#   Temporal blocks copied verbatim from the synthetic-trained model
#   so the synthetic checkpoint loads with matching shapes.
# ============================================================

class ConvLSTMCell(nn.Module):
    def __init__(self, input_dim, hidden_dim, kernel_size=3):
        super().__init__(); self.hidden_dim = hidden_dim; p = kernel_size // 2
        self.conv = nn.Conv2d(input_dim + hidden_dim, 4*hidden_dim, kernel_size, padding=p)
        self.bn   = nn.BatchNorm2d(4*hidden_dim)
    def forward(self, x, states):
        h, c = states
        i, f, o, g = torch.chunk(self.bn(self.conv(torch.cat([x, h], 1))), 4, 1)
        i, f, o, g = torch.sigmoid(i), torch.sigmoid(f), torch.sigmoid(o), torch.tanh(g)
        c_next = f*c + i*g
        return o*torch.tanh(c_next), c_next

class SEBlock(nn.Module):
    def __init__(self, ch, reduction=4):
        super().__init__(); self.pool = nn.AdaptiveAvgPool2d(1)
        self.fc = nn.Sequential(nn.Linear(ch, max(ch//reduction, 4)), nn.ReLU(),
                                nn.Linear(max(ch//reduction, 4), ch), nn.Sigmoid())
    def forward(self, x):
        B, C, _, _ = x.shape
        return x * self.fc(self.pool(x).view(B, C)).view(B, C, 1, 1)

class MotionResBlock(nn.Module):
    def __init__(self, in_ch, out_ch, stride=1):
        super().__init__(); mid = out_ch*2
        self.pw1 = nn.Sequential(nn.Conv2d(in_ch, mid, 1, bias=False), nn.BatchNorm2d(mid), nn.Hardswish())
        self.dw  = nn.Sequential(nn.Conv2d(mid, mid, 3, stride=stride, padding=1, groups=mid, bias=False),
                                 nn.BatchNorm2d(mid), nn.Hardswish())
        self.se  = SEBlock(mid)
        self.pw2 = nn.Sequential(nn.Conv2d(mid, out_ch, 1, bias=False), nn.BatchNorm2d(out_ch))
        self.res = (stride == 1 and in_ch == out_ch)
    def forward(self, x):
        out = self.pw2(self.se(self.dw(self.pw1(x))))
        return out + x if self.res else out

class MotionCNN(nn.Module):
    def __init__(self):
        super().__init__()
        self.stem   = nn.Sequential(nn.Conv2d(2, 16, 3, 2, 1, bias=False), nn.BatchNorm2d(16), nn.Hardswish())
        self.blocks = nn.Sequential(
            MotionResBlock(16, 24, 2), MotionResBlock(24, 32, 2), MotionResBlock(32, 48, 1),
            MotionResBlock(48, 64, 2), MotionResBlock(64, 64, 1))
        self.final  = nn.Sequential(nn.Conv2d(64, 64, 1, bias=False), nn.BatchNorm2d(64),
                                    nn.Hardswish(), nn.AdaptiveAvgPool2d((7, 7)))
        self.drop   = nn.Dropout2d(0.15)
    def forward(self, x): return self.drop(self.final(self.blocks(self.stem(x))))


# ============================================================
# DILATE MOBILENETV3-SMALL TO OUTPUT-STRIDE 8
#   Robust: walk blocks, track cumulative stride; once we are at 1/8,
#   convert any further stride-2 conv to stride-1 + dilation (DeepLab).
#   -> 576-channel features AT 1/8 resolution, single stream.
# ============================================================

def dilate_to_os8(features):
    cur, dil = 1, 1
    for block in features:
        strided = [m for m in block.modules()
                   if isinstance(m, nn.Conv2d) and m.stride == (2, 2)]
        if strided:
            if cur >= OUT_STRIDE:
                dil *= 2
                for m in strided:
                    k = m.kernel_size[0]
                    m.stride = (1, 1); m.dilation = (dil, dil)
                    m.padding = (dil*(k-1)//2, dil*(k-1)//2)
            else:
                cur *= 2
    return features


def make_density_head(in_ch):
    # CSRNet-style dilated backend, BatchNorm RESTORED, final ReLU (density >= 0)
    return nn.Sequential(
        nn.Conv2d(in_ch, 128, 1, bias=False), nn.BatchNorm2d(128), nn.ReLU(inplace=True),
        nn.Conv2d(128, 128, 3, padding=2, dilation=2, bias=False), nn.BatchNorm2d(128), nn.ReLU(inplace=True),
        nn.Conv2d(128, 64,  3, padding=4, dilation=4, bias=False), nn.BatchNorm2d(64),  nn.ReLU(inplace=True),
        nn.Conv2d(64,  32,  3, padding=2, dilation=2, bias=False), nn.BatchNorm2d(32),  nn.ReLU(inplace=True),
        nn.Conv2d(32,  1,   1), nn.ReLU(inplace=True))


# ============================================================
# CROWDNET V9
# ============================================================

class CrowdNetV9(nn.Module):
    def __init__(self, use_temporal=False):
        super().__init__(); self.use_temporal = use_temporal; self.hidden_dim = 128

        if os.path.exists(MOBILENET_LOCAL):
            print(f"BACKBONE: local {MOBILENET_LOCAL}")
            mob = models.mobilenet_v3_small(weights=None)
            mob.load_state_dict(torch.load(MOBILENET_LOCAL, weights_only=True))
        else:
            print("BACKBONE: ImageNet pretrained")
            mob = models.mobilenet_v3_small(weights=models.MobileNet_V3_Small_Weights.IMAGENET1K_V1)

        feats = mob.features
        # DENSITY backbone: dilated copy (output stride 8), trainable
        self.den_backbone = dilate_to_os8(copy.deepcopy(feats))
        for p in self.den_backbone.parameters(): p.requires_grad = True

        # density head input = 576 (+128 temporal fused if enabled)
        self.density_head = make_density_head(576 + (self.hidden_dim if use_temporal else 0))

        # CLASSIFICATION / TEMPORAL backbone: original strides (7x7 @224)
        cls_feats = copy.deepcopy(feats)
        self.cls_stage1 = cls_feats[:3]     # (B,24,28,28)@224
        self.cls_stage2 = cls_feats[3:]     # (B,576,7,7)@224
        self.spatial_reduce = nn.Sequential(nn.Conv2d(576, 64, 1), nn.BatchNorm2d(64),
                                            nn.ReLU(), nn.Dropout2d(0.2))
        self.flow_branch = MotionCNN()
        self.convlstm   = ConvLSTMCell(128, self.hidden_dim)
        self.pool       = nn.AdaptiveAvgPool2d(1)
        self.classifier = nn.Sequential(nn.Linear(self.hidden_dim, 128), nn.ReLU(), nn.Dropout(0.5),
                                        nn.Linear(128, 64), nn.ReLU(), nn.Dropout(0.4),
                                        nn.Linear(64, 3))

    # ---- density (single frame, native res or crop) — SPATIAL model only ----
    def forward_density(self, x):                    # x: (B,1|3,H,W) in [0,1]
        assert not self.use_temporal, \
            "forward_density is spatial-only (head=576ch). For a temporal model " \
            "use forward_count_window (head=704ch)."
        if x.shape[1] == 1: x = x.repeat(1, 3, 1, 1)
        return self.density_head(self.den_backbone(x))

    # ---- temporal encode one frame; returns (h,c) and cls feature ----
    def _temporal_step(self, frame224, flow224, h, c):
        x = frame224.repeat(1, 3, 1, 1) if frame224.shape[1] == 1 else frame224
        s2 = self.cls_stage2(self.cls_stage1(x))     # (B,576,7,7)
        sp = self.spatial_reduce(s2)                 # (B,64,7,7)
        fl = self.flow_branch(flow224)                # (B,64,7,7)
        fu = torch.cat([sp, fl], 1)                  # (B,128,7,7)
        if h is None:
            B, _, H, W = fu.shape
            h = torch.zeros(B, self.hidden_dim, H, W, device=fu.device); c = torch.zeros_like(h)
        h, c = self.convlstm(fu, (h, c))
        return h, c

    # ---- classification over a window (Task B) ----
    def forward_classify(self, frames224, flows224):
        B, T = frames224.shape[:2]; h = c = None
        for t in range(T):
            h, c = self._temporal_step(frames224[:, t], flows224[:, t], h, c)
        return self.classifier(self.pool(h).view(B, -1))

    # ---- joint temporal counting over a window (Stage 2) ----
    def forward_count_window(self, crops, frames224, flows224):
        B, T = crops.shape[:2]; h = c = None; dens = []
        for t in range(T):
            h, c = self._temporal_step(frames224[:, t], flows224[:, t], h, c)
            f = self.den_backbone(crops[:, t].repeat(1, 3, 1, 1))     # (B,576,h,w)
            if self.use_temporal:
                hup = F.interpolate(h, size=f.shape[-2:], mode="bilinear", align_corners=False)
                f = torch.cat([f, hup], 1)
            dens.append(self.density_head(f))
        return torch.stack(dens, 1)                                   # (B,T,1,h,w)


# ============================================================
# SYNTHETIC WEIGHT LOADER  (grayscale-trained checkpoint)
#   Maps backbone_stage1/2 -> den_backbone (full features) AND
#   -> cls_stage1/2, plus spatial_reduce/flow_branch/convlstm/classifier.
# ============================================================

def load_synthetic(model, ckpt_path=SYNTH_CKPT):
    if not os.path.exists(ckpt_path):
        print(f"WARNING: {ckpt_path} not found — using ImageNet/local init."); return
    ck = torch.load(ckpt_path, map_location="cpu", weights_only=True)
    def sub(prefix):
        return {k[len(prefix)+1:]: v for k, v in ck.items() if k.startswith(prefix + ".")}
    # NOTE: nn.Sequential slicing KEEPS original indices — backbone_stage1 keys
    # are "0..2", backbone_stage2 keys are "3..12".  So the full-features dict is
    # just their union (NO index offset), and each slice maps 1:1 by construction.
    s1, s2 = sub("backbone_stage1"), sub("backbone_stage2")

    # temporal / classification path (identical module defs -> exact load)
    if s1: model.cls_stage1.load_state_dict(s1, strict=True)
    if s2: model.cls_stage2.load_state_dict(s2, strict=True)
    for name in ["spatial_reduce", "flow_branch", "convlstm", "classifier"]:
        sd = sub(name)
        if sd:
            getattr(model, name).load_state_dict(sd, strict=True)

    # density backbone (dilated copy): dilation does not change param shapes,
    # so the same union of keys loads cleanly ("0..12").
    full = dict(s1); full.update(s2)
    miss, unexp = model.den_backbone.load_state_dict(full, strict=False)
    # the ONLY thing NOT loaded from synth: the new density_head (24-ch head in
    # the checkpoint is deliberately discarded — we replaced it with the dilated head).
    assert len(unexp) == 0, f"unexpected keys in den_backbone load: {unexp[:5]}"
    print(f"SYNTHETIC: den_backbone loaded (missing={len(miss)}, unexpected={len(unexp)}); "
          f"cls_stage1/2 + spatial_reduce + flow_branch + convlstm + classifier loaded strict; "
          f"density_head = fresh (by design).")


# ---- self-test: dilated backbone must output 576ch @ 1/8 ----
def _selftest_model():
    m = CrowdNetV9(use_temporal=False)
    with torch.no_grad():
        o = m.den_backbone(torch.zeros(1, 3, CROP, CROP))
    assert o.shape[1] == 576 and o.shape[2] == CROP // 8 and o.shape[3] == CROP // 8, \
        f"dilation failed, got {tuple(o.shape)} expected (1,576,{CROP//8},{CROP//8})"
    with torch.no_grad():
        d = m.forward_density(torch.zeros(1, 1, CROP, CROP))
    assert d.shape == (1, 1, CROP // 8, CROP // 8), d.shape
    print(f"SELFTEST ok: den feature {tuple(o.shape)} -> density {tuple(d.shape)}")

In [5]:
# ============================================================
# STAGE 1 : SPATIAL DENSITY TRAINING  (per-frame, native crops)
#   Loss = MSE(density) + lambda * L1(count).  BatchNorm active.
#   Differential LR: pretrained backbone gentle, random head full.
# ============================================================

def stage1_optimizer(model):
    opt = optim.AdamW(
        [{"params": model.den_backbone.parameters(), "lr": LR * 0.3, "name": "den_backbone"},
         {"params": model.density_head.parameters(), "lr": LR,       "name": "density_head"}],
        weight_decay=WEIGHT_DECAY)
    for g in opt.param_groups:
        print(f"  {g['name']:<14} LR={g['lr']:.1e}  "
              f"params={sum(p.numel() for p in g['params']):,}")
    sched = optim.lr_scheduler.CosineAnnealingLR(opt, T_max=EPOCHS, eta_min=1e-6)
    return opt, sched


def density_loss(pred, target, counts):
    # per-pixel MSE (sum, averaged over batch) + count anchor
    B = pred.shape[0]
    l_pix   = F.mse_loss(pred, target, reduction="sum") / B
    l_count = F.l1_loss(pred.sum(dim=[1, 2, 3]), counts)
    return l_pix + COUNT_LOSS_WEIGHT * l_count, l_count


def train_stage1(model):
    tr = FrameDensityDataset("train", training=True,  frame_stride=TRAIN_FRAME_STRIDE, aug_prob=0.3)
    vl = FrameDensityDataset("val",   training=False, frame_stride=1)
    tl = DataLoader(tr, BATCH_SIZE, shuffle=True,  num_workers=NUM_WORKERS, pin_memory=True, drop_last=True)
    vloader = DataLoader(vl, 1, shuffle=False, num_workers=NUM_WORKERS)

    load_synthetic(model)
    print("\nOptimizer:"); opt, sched = stage1_optimizer(model)
    model.to(device)

    best = 1e9; wait = 0
    for ep in range(1, EPOCHS + 1):
        tr.aug_prob = 0.3 if ep <= 5 else (0.5 if ep <= 20 else 0.7)
        print(f"\n{'='*60}\nEPOCH {ep}/{EPOCHS}\n{'='*60}")
        model.train()
        run_loss = run_cae = 0.0; nb = nf = 0
        loop = tqdm(tl, leave=True)
        for frame, dmap, count in loop:
            frame, dmap, count = frame.to(device), dmap.to(device), count.to(device)
            opt.zero_grad()
            pred = model.forward_density(frame)
            loss, lc = density_loss(pred, dmap, count)
            loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            opt.step()
            cae = (pred.sum(dim=[1,2,3]) - count).abs().sum().item()
            run_loss += loss.item(); run_cae += cae; nb += 1; nf += frame.shape[0]
            loop.set_description(f"TRAIN ep{ep} aug={tr.aug_prob:.1f} | "
                                 f"Loss {run_loss/nb:.3f} | MAE {run_cae/nf:.2f}")
        sched.step()

        # ---- validation (full-frame per-frame MAE) ----
        model.eval(); mae = mse = 0.0; n = 0
        with torch.no_grad():
            for frame, dmap, count in tqdm(vloader, desc="VAL", leave=False):
                pred = model.forward_density(frame.to(device))
                pc = pred.sum().item(); d = pc - count.item()
                mae += abs(d); mse += d*d; n += 1
        vmae, vmse = mae/n, math.sqrt(mse/n)
        print(f"VAL | MAE {vmae:.2f} | MSE {vmse:.2f} | LR {opt.param_groups[0]['lr']:.2e}")
        if vmae < best:
            best = vmae; wait = 0
            torch.save(model.state_dict(), BEST_MODEL)
            print(f"IMPROVED -> saved (val MAE={vmae:.2f})")
        else:
            wait += 1; print(f"no improvement ({wait}/{PATIENCE})")
            if wait >= PATIENCE * 2:
                print("EARLY STOP"); break
    print(f"\nBest val MAE: {best:.2f}")
    model.load_state_dict(torch.load(BEST_MODEL, map_location=device))
    return model

In [6]:
# ============================================================
# TASK A : DENSITY EVALUATION  (full-frame, per-frame MAE)
#   Fully-convolutional: run the whole 1080x1920 frame, sum the
#   1/8 density -> predicted count.  Matches the DroneCrowd protocol.
# ============================================================

def evaluate_taskA(model):
    model.eval().to(device)
    mae = mse = 0.0; n = 0
    with torch.no_grad():
        if not model.use_temporal:
            # SPATIAL model: full-frame fully-convolutional, one frame at a time.
            loader = DataLoader(FrameDensityDataset("test", training=False, frame_stride=1),
                                1, shuffle=False, num_workers=NUM_WORKERS)
            for frame, dmap, count in tqdm(loader, desc="EVAL"):
                pred = model.forward_density(frame.to(device))
                d = pred.sum().item() - count.item()
                mae += abs(d); mse += d*d; n += 1
        else:
            # TEMPORAL model: windowed, native full frames through the fused head.
            loader = DataLoader(WindowDataset("test", mode="count", training=False),
                                1, shuffle=False, num_workers=NUM_WORKERS)
            for crops, frames224, flows224, dens, counts in tqdm(loader, desc="EVAL"):
                pred = model.forward_count_window(crops.to(device), frames224.to(device),
                                                  flows224.to(device))   # (1,T,1,h,w)
                pc = pred.sum(dim=[2, 3, 4]).cpu().view(-1)               # (T,)
                gt = counts.view(-1)
                d = (pc - gt).numpy()
                mae += abs(d).sum(); mse += (d*d).sum(); n += d.size
    mae, mse = mae/n, math.sqrt(mse/n)
    print("\n" + "="*65)
    print("TASK A — CROWD COUNTING (all test frames, per-frame MAE)")
    print("="*65)
    print(f"\n  {'Method':<32}{'MAE':>10}{'MSE':>10}")
    print("  " + "-"*52)
    for name, m, s in [("STNNet  (Wen, CVPR 2021)", 25.14, 43.04),
                       ("STANet  (Wen, arXiv 2021)", 21.15, 35.19),
                       ("DenseTrack (2024)", 18.31, 29.68),
                       ("CrowdNet V9 (ours)", mae, mse)]:
        arrow = " <-" if "ours" in name else ""
        print(f"  {name:<32}{m:>10.2f}{s:>10.2f}{arrow}")
    print("="*65)
    return mae, mse

In [7]:
# ============================================================
# TASK B : QUALITATIVE BEHAVIOUR CLASSIFICATION
#   Always uses a FROZEN SYNTHETIC SNAPSHOT (fresh model + synthetic
#   weights), so Stage-1/2 density fine-tuning can never corrupt it.
# ============================================================

def evaluate_taskB():
    snap = CrowdNetV9(use_temporal=False)
    load_synthetic(snap)                 # frozen synthetic snapshot
    snap.eval().to(device)

    root = os.path.join(PROCESSED_DIR, "test_data")
    seq_names = sorted(os.listdir(root))
    dist = defaultdict(lambda: {0: 0, 1: 0, 2: 0}); avg_gt = {}

    for seq_name in seq_names:
        seq_dir = os.path.join(root, seq_name)
        gt = np.load(os.path.join(seq_dir, "gt_counts.npy")); avg_gt[seq_name] = float(gt.mean())
        # per-seq classify dataset (reuse WindowDataset by filtering to one seq)
        ds = WindowDataset.__new__(WindowDataset)
        ds.split_disk = "test_data"; ds.mode = "classify"; ds.training = False
        ds.seq_length = SEQ_LENGTH
        fids = np.load(os.path.join(seq_dir, "frame_ids.npy")); seq_id = int(seq_name.replace("seq",""))
        ds.windows = [(seq_dir, seq_id, s, fids)
                      for s in range(WARMUP_FRAMES, len(fids) - SEQ_LENGTH, STRIDE_W)]
        if not ds.windows: continue
        loader = DataLoader(ds, 4, shuffle=False, num_workers=NUM_WORKERS)
        with torch.no_grad():
            for frames224, flows224 in loader:
                logits = snap.forward_classify(frames224.to(device), flows224.to(device))
                for p in torch.argmax(logits, 1).cpu().numpy():
                    dist[seq_name][int(p)] += 1

    print("\n" + "="*75)
    print("TASK B — QUALITATIVE BEHAVIOUR CLASSIFICATION [synthetic snapshot]")
    print("="*75)
    print(f"{'Sequence':<12}{'AvgCount':>9}{'Attr':>9}  {'Normal%':>8}{'Bottl%':>8}{'Panic%':>8}  Dominant")
    print("-"*75)
    for seq_name in sorted(dist.keys()):
        d = dist[seq_name]; tot = max(sum(d.values()), 1)
        attr = "Crowded" if avg_gt[seq_name] > CROWDED_THRESH else "Sparse"
        dom = CLASS_NAMES[max(d, key=d.get)]
        print(f"{seq_name:<12}{avg_gt[seq_name]:>9.1f}{attr:>9}  "
              f"{100*d[0]/tot:>7.1f}%{100*d[1]/tot:>7.1f}%{100*d[2]/tot:>7.1f}%  {dom}")
    print("="*75)
    return dist

In [8]:
# ============================================================
# TASK C : MOTION-SIGNAL PROBE — patch-averaged flow vs GT trajectory
#   Honest test of whether the flow field (MotionCNN's INPUT) carries
#   usable directional signal per person.  Computed at TEMP_SIZE (224),
#   trajectories scaled from native to 224.
# ============================================================

def patch_flow(flow, px, py, patch=16):
    H, W = flow.shape[:2]
    x0, x1 = max(0, px-patch), min(W, px+patch+1)
    y0, y1 = max(0, py-patch), min(H, py+patch+1)
    return flow[y0:y1, x0:x1, :].reshape(-1, 2).mean(0)

def evaluate_taskC(max_seqs=TASK_C_MAX_SEQS, max_frames=TASK_C_MAX_FRAMES):
    root = os.path.join(PROCESSED_DIR, "test_data")
    seq_names = sorted(os.listdir(root))[:max_seqs]
    sx, sy = TEMP_SIZE / IMG_W, TEMP_SIZE / IMG_H
    cos_sum = ang_sum = 0.0; n = 0; bins = np.zeros(7, int); n_close15 = n_close30 = 0
    for seq_name in tqdm(seq_names, desc="Task C"):
        seq_dir = os.path.join(root, seq_name)
        fids = np.load(os.path.join(seq_dir, "frame_ids.npy"))
        seq_id = int(seq_name.replace("seq", ""))
        traj = np.load(os.path.join(seq_dir, "trajectories.npy"))     # (t, x_native, y_native)
        by_t = defaultdict(list)
        for t, x, y in traj: by_t[int(t)].append((x*sx, y*sy))
        prev = None
        for t in range(min(max_frames, len(fids))):
            g = cv2.imread(jpg_path("test_data", seq_id, int(fids[t])), cv2.IMREAD_GRAYSCALE)
            if g is None: continue
            cur = cv2.resize(g, (TEMP_SIZE, TEMP_SIZE))
            if prev is not None and t in by_t and (t-1) in by_t:
                flow = farneback(prev, cur)                            # (224,224,2)
                pts_prev = {(round(x), round(y)) for x, y in by_t[t-1]}
                for (x, y) in by_t[t]:
                    # GT displacement: nearest prev point (approx association)
                    gt = None; best = 1e9
                    for (px, py) in by_t[t-1]:
                        dd = (px-x)**2 + (py-y)**2
                        if dd < best: best, gt = dd, (x-px, y-py)
                    if gt is None: continue
                    gmag = math.hypot(*gt)
                    if gmag < 0.3: continue
                    fv = patch_flow(flow, int(np.clip(x,0,223)), int(np.clip(y,0,223)))
                    fmag = math.hypot(*fv)
                    if fmag < 1e-6: continue
                    cs = (gt[0]*fv[0] + gt[1]*fv[1]) / (gmag*fmag)
                    cs = float(np.clip(cs, -1, 1)); ang = math.degrees(math.acos(cs))
                    cos_sum += cs; ang_sum += ang; n += 1
                    bins[min(int(ang//15), 6)] += 1
                    n_close15 += ang < 15; n_close30 += ang < 30
            prev = cur
    if n == 0: print("Task C: no motion evaluated."); return
    print("\n" + "="*65)
    print("TASK C — PATCH-AVERAGED FLOW vs GT TRAJECTORY")
    print("="*65)
    print(f"  Persons evaluated      : {n:,}")
    print(f"  Mean cosine similarity : {cos_sum/n:.4f}  (1=perfect, 0=random)")
    print(f"  Mean angle error       : {ang_sum/n:.2f} deg  (random ~90)")
    print(f"  Fraction < 15 deg      : {100*n_close15/n:.1f}%  (random ~8.6%)")
    print(f"  Fraction < 30 deg      : {100*n_close30/n:.1f}%  (random ~17%)")
    verdict = "SIGNAL DETECTED" if (ang_sum/n) < 80 else "NO CLEAR SIGNAL"
    print(f"  Verdict: {verdict}")
    print("="*65)

In [13]:
# ============================================================
# STAGE 2 (OPTIONAL) : TEMPORAL COUNT REFINEMENT
#   Requires USE_TEMPORAL_COUNT=True.  Temporal branch is UNFROZEN and
#   fine-tuned end-to-end; its state is fused into the density head.
#   Task B still uses the frozen synthetic snapshot (evaluate_taskB).
#
#   Init: synthetic weights (all branches) + Stage-1 density backbone.
#   Loss: MSE(density) + lambda*L1(count) + mu*temporal-count-smoothness
#         (count should change slowly when scene motion is low).
# ============================================================

def stage2_optimizer(model):
    groups = [
        {"params": model.den_backbone.parameters(),   "lr": LR*0.3,  "name": "den_backbone"},
        {"params": model.density_head.parameters(),   "lr": LR,      "name": "density_head"},
        {"params": model.cls_stage1.parameters(),     "lr": LR*0.1,  "name": "cls_stage1"},
        {"params": model.cls_stage2.parameters(),     "lr": LR*0.1,  "name": "cls_stage2"},
        {"params": model.spatial_reduce.parameters(), "lr": LR*0.3,  "name": "spatial_reduce"},
        {"params": model.flow_branch.parameters(),     "lr": LR*0.3,  "name": "flow_branch"},
        {"params": model.convlstm.parameters(),       "lr": LR*0.3,  "name": "convlstm"},
    ]
    for p in model.classifier.parameters(): p.requires_grad = False   # keep cls head for snapshot parity
    opt = optim.AdamW(groups, weight_decay=WEIGHT_DECAY)
    for g in opt.param_groups:
        print(f"  {g['name']:<15} LR={g['lr']:.1e}  params={sum(p.numel() for p in g['params']):,}")
    return opt, optim.lr_scheduler.CosineAnnealingLR(opt, T_max=EPOCHS, eta_min=1e-6)


def train_stage2():
    assert USE_TEMPORAL_COUNT, "Set USE_TEMPORAL_COUNT=True (config cell) before Stage 2."
    model = CrowdNetV9(use_temporal=True)
    load_synthetic(model)
    if os.path.exists(BEST_MODEL):
        sd = torch.load(BEST_MODEL, map_location="cpu")
        bb = {k[len("den_backbone."):]: v for k, v in sd.items() if k.startswith("den_backbone.")}
        model.den_backbone.load_state_dict(bb, strict=False)
        print("Loaded Stage-1 density backbone into temporal model.")
    model.to(device)

    tr = WindowDataset("train", mode="count", training=True)
    vl = WindowDataset("val",   mode="count", training=False)
    tl = DataLoader(tr, max(2, BATCH_SIZE//4), shuffle=True,  num_workers=NUM_WORKERS, pin_memory=True, drop_last=True)
    vloader = DataLoader(vl, 1, shuffle=False, num_workers=NUM_WORKERS)

    print("\nStage-2 optimizer:"); opt, sched = stage2_optimizer(model)
    best = 1e9; wait = 0
    for ep in range(1, EPOCHS + 1):
        print(f"\n{'='*60}\nSTAGE2 EPOCH {ep}/{EPOCHS}\n{'='*60}")
        model.train()
        run = cae_run = 0.0; nb = nf = 0
        for crops, frames224, flows224, dens, counts in tqdm(tl, leave=True):
            crops, frames224, flows224 = crops.to(device), frames224.to(device), flows224.to(device)
            dens, counts = dens.to(device), counts.to(device)          # dens (B,T,1,h,w), counts (B,T)
            opt.zero_grad()
            pred = model.forward_count_window(crops, frames224, flows224)   # (B,T,1,h,w)
            B, T = counts.shape
            l_pix   = F.mse_loss(pred, dens, reduction="sum") / (B*T)
            pcount  = pred.sum(dim=[2, 3, 4])                           # (B,T)
            l_count = F.l1_loss(pcount, counts)
            fmag    = flows224.abs().mean(dim=[2, 3, 4])                # (B,T) scene motion proxy
            dcount  = (pcount[:, 1:] - pcount[:, :-1]).abs()
            l_temp  = (dcount * (1.0 - fmag[:, 1:].clamp(0, 1))).mean()
            loss = l_pix + COUNT_LOSS_WEIGHT*l_count + TEMP_SMOOTH_WEIGHT*l_temp
            loss.backward(); torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0); opt.step()
            cae_run += (pcount - counts).abs().sum().item(); run += loss.item()
            nb += 1; nf += B*T
        sched.step()

        model.eval(); mae = 0.0; n = 0
        with torch.no_grad():
            for crops, frames224, flows224, dens, counts in tqdm(vloader, desc="VAL", leave=False):
                pred = model.forward_count_window(crops.to(device), frames224.to(device), flows224.to(device))
                pc = pred.sum(dim=[2,3,4]).cpu()                       # (1,T)
                mae += (pc - counts).abs().sum().item(); n += counts.numel()
        vmae = mae/n
        print(f"TRAIN MAE {cae_run/nf:.2f} | VAL MAE {vmae:.2f}")
        if vmae < best:
            best = vmae; wait = 0; torch.save(model.state_dict(), "best_dronecrowd_v9_temporal.pth")
            print(f"IMPROVED -> saved (val MAE={vmae:.2f})")
        else:
            wait += 1
            if wait >= PATIENCE*2: print("EARLY STOP"); break
    print(f"\nStage-2 best val MAE: {best:.2f}")
    model.load_state_dict(torch.load("best_dronecrowd_v9_temporal.pth", map_location=device))
    return model

In [10]:
# ============================================================
# MAIN DRIVER
# ============================================================
# STAGE 1 (recommended first — pure spatial density -> aim MAE < 30):
#
#   preprocess_all()
#   sanity_check()
#   _selftest_model()                       # verifies dilation -> 576ch @ 1/8
#   model = CrowdNetV9(use_temporal=False)
#   model = train_stage1(model)
#   evaluate_taskA(model)
#   evaluate_taskB()                        # frozen synthetic snapshot
#   evaluate_taskC()
#
# STAGE 2 (only after Stage 1 works — temporal count refinement):
#   1) In the CONFIG cell set:  USE_TEMPORAL_COUNT = True
#   2) Re-run the config + model cells, then:
#        model = train_stage2()
#        # Task A on the temporal model:
#        evaluate_taskA(model)              # uses forward_density; for full
#        #  temporal-fused counting use forward_count_window over windows.
#
# NOTE on the fair grayscale comparison you want:
#   To compare STNNet / CSRNet on grayscale, feed them the same single
#   channel replicated to 3 (as here) and rebuild their GT density with
#   build_density_native (sigma=4, 1/8).  Keep MODEL crops/native res
#   identical.  Then the only difference is the network, which is the
#   comparison you're after.
# ============================================================

preprocess_all()
sanity_check()
_selftest_model()                       # verifies dilation -> 576ch @ 1/8

if not USE_TEMPORAL_COUNT:
    model = CrowdNetV9(use_temporal=False)
    model = train_stage1(model)         # Stage 1: spatial density
else:
    model = train_stage2()              # Stage 2: temporal count refinement

evaluate_taskA(model)                   # per-frame counting MAE/MSE
evaluate_taskB()                        # behaviour (frozen synthetic snapshot)
evaluate_taskC()                        # motion-signal probe
print("\n" + "="*60 + "\nPIPELINE COMPLETE\n" + "="*60)


Pre-processing: train_data
  [train_data] 82 sequences (24600 frames)  [min_frames=20]


  preprocess train_data: 100%|████████████████████████████████████████████████████████| 82/82 [00:00<00:00, 529.22it/s]



Pre-processing: test_data
  [test_data] 30 sequences (9000 frames)  [min_frames=20]


  preprocess test_data: 100%|█████████████████████████████████████████████████████████| 30/30 [00:00<00:00, 499.68it/s]



Pre-processing: val_data
  [val_data] 30 sequences (360 frames)  [min_frames=12]


...

VAL | MAE 23.40 | MSE 28.72 | LR 8.25e-06
no improvement (24/12)
EARLY STOP

Best val MAE: 17.96
  FrameDensity[test] 9000 frames (training=False, stride=1)


EVAL: 100%|████████████████████████████████████████████████████████████████████████| 9000/9000 [07:11<00:00, 20.86it/s]



TASK A — CROWD COUNTING (all test frames, per-frame MAE)

  Method                                 MAE       MSE
  ----------------------------------------------------
  STNNet  (Wen, CVPR 2021)             25.14     43.04
  STANet  (Wen, arXiv 2021)            21.15     35.19
  DenseTrack (2024)                    18.31     29.68
  CrowdNet V9 (ours)                   18.86     23.58 <-
BACKBONE: local mobilenet_v3_small-047dcff4.pth
SYNTHETIC: den_backbone loaded (missing=0, unexpected=0); cls_stage1/2 + spatial_reduce + flow_branch + convlstm + classifier loaded strict; density_head = fresh (by design).

TASK B — QUALITATIVE BEHAVIOUR CLASSIFICATION [synthetic snapshot]
Sequence     AvgCount     Attr   Normal%  Bottl%  Panic%  Dominant
---------------------------------------------------------------------------
seq011          205.5  Crowded     81.0%   15.5%    3.4%  Normal
seq015          148.6   Sparse      0.0%  100.0%    0.0%  Bottleneck
seq016          146.2   Sparse      0.0%

Task C: 100%|████████████████████████████████████████████████████████████████████████████| 5/5 [00:10<00:00,  2.14s/it]


TASK C — PATCH-AVERAGED FLOW vs GT TRAJECTORY
  Persons evaluated      : 778
  Mean cosine similarity : 0.5957  (1=perfect, 0=random)
  Mean angle error       : 44.56 deg  (random ~90)
  Fraction < 15 deg      : 30.8%  (random ~8.6%)
  Fraction < 30 deg      : 51.5%  (random ~17%)
  Verdict: SIGNAL DETECTED

PIPELINE COMPLETE


In [12]:
# ============================================================
# MAIN DRIVER
# ============================================================
# STAGE 1 (recommended first — pure spatial density -> aim MAE < 30):
#
#   preprocess_all()
#   sanity_check()
#   _selftest_model()                       # verifies dilation -> 576ch @ 1/8
#   model = CrowdNetV9(use_temporal=False)
#   model = train_stage1(model)
#   evaluate_taskA(model)
#   evaluate_taskB()                        # frozen synthetic snapshot
#   evaluate_taskC()
#
# STAGE 2 (only after Stage 1 works — temporal count refinement):
#   1) In the CONFIG cell set:  USE_TEMPORAL_COUNT = True
#   2) Re-run the config + model cells, then:
#        model = train_stage2()
#        # Task A on the temporal model:
#        evaluate_taskA(model)              # uses forward_density; for full
#        #  temporal-fused counting use forward_count_window over windows.
#
# NOTE on the fair grayscale comparison you want:
#   To compare STNNet / CSRNet on grayscale, feed them the same single
#   channel replicated to 3 (as here) and rebuild their GT density with
#   build_density_native (sigma=4, 1/8).  Keep MODEL crops/native res
#   identical.  Then the only difference is the network, which is the
#   comparison you're after.
# ============================================================

preprocess_all()
sanity_check()
_selftest_model()                       # verifies dilation -> 576ch @ 1/8

if not USE_TEMPORAL_COUNT:
    model = CrowdNetV9(use_temporal=False)
    model = train_stage1(model)         # Stage 1: spatial density
else:
    model = train_stage2()              # Stage 2: temporal count refinement

evaluate_taskA(model)                   # per-frame counting MAE/MSE
evaluate_taskB()                        # behaviour (frozen synthetic snapshot)
evaluate_taskC()                        # motion-signal probe
print("\n" + "="*60 + "\nPIPELINE COMPLETE\n" + "="*60)


Pre-processing: train_data
  [train_data] 82 sequences (24600 frames)  [min_frames=20]


  preprocess train_data: 100%|████████████████████████████████████████████████████████| 82/82 [00:00<00:00, 282.43it/s]



Pre-processing: test_data
  [test_data] 30 sequences (9000 frames)  [min_frames=20]


  preprocess test_data: 100%|█████████████████████████████████████████████████████████| 30/30 [00:00<00:00, 563.27it/s]



Pre-processing: val_data
  [val_data] 30 sequences (360 frames)  [min_frames=12]


...

TRAIN MAE 2.51 | VAL MAE 35.31
EARLY STOP

Stage-2 best val MAE: 19.56
  Window[test/count] 1740 windows (T=10)


EVAL: 100%|████████████████████████████████████████████████████████████████████████| 1740/1740 [19:46<00:00,  1.47it/s]



TASK A — CROWD COUNTING (all test frames, per-frame MAE)

  Method                                 MAE       MSE
  ----------------------------------------------------
  STNNet  (Wen, CVPR 2021)             25.14     43.04
  STANet  (Wen, arXiv 2021)            21.15     35.19
  DenseTrack (2024)                    18.31     29.68
  CrowdNet V9 (ours)                   36.88     43.09 <-
BACKBONE: local mobilenet_v3_small-047dcff4.pth
SYNTHETIC: den_backbone loaded (missing=0, unexpected=0); cls_stage1/2 + spatial_reduce + flow_branch + convlstm + classifier loaded strict; density_head = fresh (by design).

TASK B — QUALITATIVE BEHAVIOUR CLASSIFICATION [synthetic snapshot]
Sequence     AvgCount     Attr   Normal%  Bottl%  Panic%  Dominant
---------------------------------------------------------------------------
seq011          205.5  Crowded     81.0%   15.5%    3.4%  Normal
seq015          148.6   Sparse      0.0%  100.0%    0.0%  Bottleneck
seq016          146.2   Sparse      0.0%

Task C: 100%|████████████████████████████████████████████████████████████████████████████| 5/5 [00:08<00:00,  1.63s/it]


TASK C — PATCH-AVERAGED FLOW vs GT TRAJECTORY
  Persons evaluated      : 778
  Mean cosine similarity : 0.5957  (1=perfect, 0=random)
  Mean angle error       : 44.56 deg  (random ~90)
  Fraction < 15 deg      : 30.8%  (random ~8.6%)
  Fraction < 30 deg      : 51.5%  (random ~17%)
  Verdict: SIGNAL DETECTED

PIPELINE COMPLETE


In [1]:
# ============================================================
# TASK C — STANDALONE (no model, no training, no re-preprocessing)
#   Hardened: TRUE track-ID association, ALL 30 test sequences, all frames.
#   Reads: dronecrowd/test_data/{images,ground_truth} + processed frame_ids.npy
#   Just paste this whole cell and run:  evaluate_taskC()
# ============================================================
import os, math, cv2, numpy as np, scipy.io
from collections import defaultdict
from tqdm.auto import tqdm
cv2.setNumThreads(0)

# ---- config (must match your run) ----
DRONECROWD_ROOT   = "dronecrowd"
PROCESSED_DIR     = "dronecrowd_processed_v9"
IMG_W, IMG_H      = 1920, 1080
TEMP_SIZE         = 224
TASK_C_MAX_SEQS   = None    # None = all 30 test sequences
TASK_C_MAX_FRAMES = None    # None = all frames per sequence

# ---- helpers ----
def image_name(seq_id, frame_id): return f"img{seq_id:03d}{frame_id:03d}.jpg"
def gt_name(seq_id, frame_id):    return f"GT_img{seq_id:03d}{frame_id:03d}.mat"
def jpg_path(split_disk, seq_id, fid):
    return os.path.join(DRONECROWD_ROOT, split_disk, "images", image_name(seq_id, fid))

def farneback(prev, cur):
    fl = cv2.calcOpticalFlowFarneback(prev, cur, None, 0.5, 3, 15, 3, 5, 1.2, 0)
    return np.clip(fl / 8.0, -1.0, 1.0).astype(np.float32)     # (H,W,2) in [-1,1]

def patch_flow(flow, px, py, patch=16):
    H, W = flow.shape[:2]
    x0, x1 = max(0, px-patch), min(W, px+patch+1)
    y0, y1 = max(0, py-patch), min(H, py+patch+1)
    return flow[y0:y1, x0:x1, :].reshape(-1, 2).mean(0)

def read_mat_tracks(mat_path):
    # returns ({track_id:(x_native,y_native)}, has_true_ids)
    loc = scipy.io.loadmat(mat_path)["image_info"][0, 0]["location"][0, 0].astype(np.float32)
    if loc.shape[1] >= 3:
        return {int(r[2]): (float(r[0]), float(r[1])) for r in loc}, True
    return {i: (float(r[0]), float(r[1])) for i, r in enumerate(loc)}, False

# ---- Task C ----
def evaluate_taskC(max_seqs=TASK_C_MAX_SEQS, max_frames=TASK_C_MAX_FRAMES,
                   patch=16, stationary_thresh=0.3):
    root   = os.path.join(PROCESSED_DIR, "test_data")
    gt_dir = os.path.join(DRONECROWD_ROOT, "test_data", "ground_truth")
    seq_names = sorted(os.listdir(root))
    if max_seqs is not None: seq_names = seq_names[:max_seqs]
    sx, sy = TEMP_SIZE / IMG_W, TEMP_SIZE / IMG_H
    cos_sum = 0.0; angs = []; n = 0; n_seq = 0; true_ids_everywhere = True

    for seq_name in tqdm(seq_names, desc="Task C"):
        seq_dir = os.path.join(root, seq_name)
        fids    = np.load(os.path.join(seq_dir, "frame_ids.npy"))
        seq_id  = int(seq_name.replace("seq", ""))
        nfr     = len(fids) if max_frames is None else min(max_frames, len(fids))
        prev_gray = None; prev_tracks = None; n_seq += 1
        for t in range(nfr):
            g = cv2.imread(jpg_path("test_data", seq_id, int(fids[t])), cv2.IMREAD_GRAYSCALE)
            if g is None: prev_gray = None; prev_tracks = None; continue
            cur = cv2.resize(g, (TEMP_SIZE, TEMP_SIZE))
            mp  = os.path.join(gt_dir, gt_name(seq_id, int(fids[t])))
            cur_tracks, has_ids = read_mat_tracks(mp) if os.path.exists(mp) else ({}, True)
            true_ids_everywhere &= has_ids
            if prev_gray is not None and prev_tracks:
                flow = farneback(prev_gray, cur)                     # prev->cur, @224
                for tid, (x, y) in cur_tracks.items():
                    if tid not in prev_tracks: continue
                    px, py = prev_tracks[tid]
                    dx, dy = (x - px) * sx, (y - py) * sy            # TRUE displacement @224
                    gmag = math.hypot(dx, dy)
                    if gmag < stationary_thresh: continue
                    fv = patch_flow(flow, int(np.clip(px*sx, 0, TEMP_SIZE-1)),
                                          int(np.clip(py*sy, 0, TEMP_SIZE-1)), patch)   # @PREV pos
                    fmag = math.hypot(*fv)
                    if fmag < 1e-6: continue
                    cs = float(np.clip((dx*fv[0] + dy*fv[1]) / (gmag*fmag), -1, 1))
                    cos_sum += cs; angs.append(math.degrees(math.acos(cs))); n += 1
            prev_gray = cur; prev_tracks = cur_tracks

    if n == 0: print("Task C: no moving persons evaluated."); return
    angs = np.array(angs)
    bins = np.array([np.sum((angs >= a) & (angs < a+15)) for a in range(0, 90, 15)]
                    + [np.sum((angs >= 90) & (angs < 135)), np.sum(angs >= 135)])
    f15, f30, f45 = (angs < 15).mean(), (angs < 30).mean(), (angs < 45).mean()
    print("\n" + "="*68)
    print("TASK C — PATCH-AVERAGED FLOW vs TRUE GT TRAJECTORY (track-ID matched)")
    print("="*68)
    print(f"  Association            : {'TRUE track-IDs' if true_ids_everywhere else 'FALLBACK (no ids in .mat!)'}")
    print(f"  Sequences / persons    : {n_seq} seqs · {n:,} moving persons")
    print(f"  Mean cosine similarity : {cos_sum/n:.4f}   (1=perfect, 0=random, -1=opposite)")
    print(f"  Mean angle error       : {angs.mean():.2f} deg   (random baseline 90.0)")
    print(f"  Median angle error     : {np.median(angs):.2f} deg")
    print(f"  Fraction < 15 deg      : {100*f15:.1f}%   (random  8.3%  -> {f15/(15/180):.1f}x chance)")
    print(f"  Fraction < 30 deg      : {100*f30:.1f}%   (random 16.7%  -> {f30/(30/180):.1f}x chance)")
    print(f"  Fraction < 45 deg      : {100*f45:.1f}%   (random 25.0%  -> {f45/(45/180):.1f}x chance)")
    print(f"  Verdict: {'SIGNAL DETECTED' if angs.mean() < 80 else 'NO CLEAR SIGNAL'}")
    print("\n  Angle-error distribution:")
    labels = ["0-15","15-30","30-45","45-60","60-75","75-90","90-135","135-180"]
    for lab, c in zip(labels, bins):
        print(f"    {lab:>7} deg  {int(c):>7}  ({100*c/n:>4.1f}%)  " + "#"*int(40*c/max(bins.max(),1)))
    print("="*68)
    return {"n": n, "mean_cos": cos_sum/n, "mean_ang": float(angs.mean()),
            "median_ang": float(np.median(angs)), "f15": float(f15), "f30": float(f30)}

evaluate_taskC()

C:\Users\RAJESH\csrnet-env\lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
Task C: 100%|██████████████████████████████████████████████████████████████████████████| 30/30 [04:42<00:00,  9.43s/it]


TASK C — PATCH-AVERAGED FLOW vs TRUE GT TRAJECTORY (track-ID matched)
  Association            : TRUE track-IDs
  Sequences / persons    : 30 seqs · 24,773 moving persons
  Mean cosine similarity : 0.3215   (1=perfect, 0=random, -1=opposite)
  Mean angle error       : 65.74 deg   (random baseline 90.0)
  Median angle error     : 51.17 deg
  Fraction < 15 deg      : 19.2%   (random  8.3%  -> 2.3x chance)
  Fraction < 30 deg      : 34.0%   (random 16.7%  -> 2.0x chance)
  Fraction < 45 deg      : 45.9%   (random 25.0%  -> 1.8x chance)
  Verdict: SIGNAL DETECTED

  Angle-error distribution:
       0-15 deg     4765  (19.2%)  ########################################
      15-30 deg     3670  (14.8%)  ##############################
      30-45 deg     2932  (11.8%)  ########################
      45-60 deg     2264  ( 9.1%)  ###################
      60-75 deg     1815  ( 7.3%)  ###############
      75-90 deg     1622  ( 6.5%)  #############
     90-135 deg     3984  (16.1%)  ############

{'n': 24773,
 'mean_cos': 0.3215419633974751,
 'mean_ang': 65.74313193364156,
 'median_ang': 51.16519005633488,
 'f15': 0.19234650627699512,
 'f30': 0.34049166431195255}